In [1]:
!pip install z3-solver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.5/36.5 MB 51.6 MB/s eta 0:00:00


In [2]:
from z3 import Solver, Int, If, Sum, sat

In [3]:
def add_feedback_constraints(solver, secret_vars, guess, black, white):
    """Applies Mastermind feedback to the Z3 solver."""
    exact_matches = [If(secret_vars[i] == guess[i], 1, 0) for i in range(len(secret_vars))]
    solver.add(Sum(exact_matches) == black)

    total_matches_exprs = []
    unique_colors = set(guess)
    for color in unique_colors:
        count_in_secret = Sum([If(secret_vars[i] == color, 1, 0) for i in range(len(secret_vars))])
        count_in_guess = guess.count(color)
        total_matches_exprs.append(If(count_in_secret < count_in_guess, count_in_secret, count_in_guess))

    solver.add(Sum(total_matches_exprs) == (black + white))

In [4]:
def get_real_feedback(guess, true_secret):
    """Simulates the Game Master returning Black and White pegs."""
    # Black pegs: Exact position and color matches
    black = sum(g == s for g, s in zip(guess, true_secret))

    # Total overlap: Minimum count of each color between guess and secret
    total_overlap = sum(min(guess.count(c), true_secret.count(c)) for c in set(guess))

    # White pegs are the remaining overlap after subtracting exact matches
    white = total_overlap - black
    return black, white

In [5]:
def play_mastermind_ai(true_secret, num_colors=6):
    """The main AI game loop."""
    num_pegs = len(true_secret)
    s = Solver()

    # 1. Initialize variables representing the unknown secret code
    secret_vars = [Int(f's_{i}') for i in range(num_pegs)]

    # 2. Add base domain constraints (Colors must be between 1 and num_colors)
    for var in secret_vars:
        s.add(var >= 1, var <= num_colors)

    print(f"True Secret Code: {true_secret}\n")

    turn = 1
    max_turns = 10

    # 3. Game Loop
    while turn <= max_turns:
        # Ask Z3 if there is any remaining valid code that satisfies all previous feedback
        if s.check() == sat:
            # Extract the AI's guess from the Z3 model
            model = s.model()
            guess = [model[secret_vars[i]].as_long() for i in range(num_pegs)]

            # Game Master evaluates the guess
            black, white = get_real_feedback(guess, true_secret)
            print(f"Turn {turn} | AI Guesses: {guess} | Feedback: {black}B, {white}W")

            # Win condition
            if black == num_pegs:
                print(f"\n🎉 AI cracked the code in {turn} turns!")
                return turn

            # Feed the new constraint back into Z3
            add_feedback_constraints(s, secret_vars, guess, black, white)
            turn += 1
        else:
            print("Error: The constraints are unsatisfiable. Something went wrong with the logic.")
            return -1

    print("\n❌ AI failed to crack the code within the turn limit.")
    return -1

In [6]:
# --- Run the Simulation with User Input ---
if __name__ == "__main__":
    try:
        user_input = input("Enter the secret code (e.g. 4, 1, 2, 6 or 4 1 2 6): ")
        # Parse inputs separated by commas or spaces
        if "," in user_input:
            secret_code = [int(x.strip()) for x in user_input.split(",") if x.strip()]
        else:
            secret_code = [int(x.strip()) for x in user_input.split() if x.strip()]

        if len(secret_code) == 0:
            print("Error: Secret code cannot be empty.")
        else:
            # Determine the maximum color value dynamically to set num_colors automatically
            num_colors = max(max(secret_code), 6)
            play_mastermind_ai(secret_code, num_colors=num_colors)
    except ValueError:
        print("Error: Please enter a valid sequence of integers.")

Enter the secret code (e.g. 4, 1, 2, 6 or 4 1 2 6): 3 2 4 5
True Secret Code: [3, 2, 4, 5]

Turn 1 | AI Guesses: [1, 1, 1, 1] | Feedback: 0B, 0W
Turn 2 | AI Guesses: [2, 2, 2, 2] | Feedback: 1B, 0W
Turn 3 | AI Guesses: [3, 3, 2, 3] | Feedback: 1B, 1W
Turn 4 | AI Guesses: [4, 2, 4, 3] | Feedback: 2B, 1W
Turn 5 | AI Guesses: [3, 2, 4, 5] | Feedback: 4B, 0W

🎉 AI cracked the code in 5 turns!
